# Store Traffic & Conversion Analytics - Synthetic Data Generator

## Overview
This notebook generates realistic synthetic data for the Store Traffic & Conversion Analytics use case in the Retail industry. The data simulates:

- **Foot Traffic Sensors**: Entry/exit counts at store entrances with timestamps
- **WiFi Probe Analytics**: Anonymized device detections for dwell time and zone tracking
- **Video Analytics Metadata**: People counts, queue lengths, and zone occupancy
- **POS Transactions**: Transaction completion events for conversion calculation
- **Store Registry**: Store metadata including format, location, and capacity

## Data Contract
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_stores` | Store registry with specifications | 100 stores |
| `dim_zones` | Zone definitions per store | ~500 zones |
| `fact_traffic_counts` | Entry/exit sensor readings | ~1M rows/month |
| `fact_wifi_probes` | WiFi device detections | ~500K rows/month |
| `fact_video_analytics` | Video-derived metrics | ~300K rows/month |
| `fact_pos_transactions` | Transaction completions | ~200K rows/month |

## How to Run in Microsoft Fabric
1. Create a new Lakehouse in your Fabric workspace
2. Attach this notebook to the Lakehouse
3. Run all cells to generate the synthetic data
4. Data will be written as Delta tables to your Lakehouse
5. For streaming simulation, schedule this notebook to run every 5 minutes

## Known Limitations
- All data is synthetic and non-identifying
- Traffic patterns follow typical retail cycles but may not match all regional variations
- WiFi MAC addresses are randomized hashes (not real device identifiers)

## Setup and Configuration

In [ ]:
# Install required packages (uncomment if needed)
!pip install faker

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import hashlib
import warnings
warnings.filterwarnings('ignore')

# Initialize Faker for realistic names and locations
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print("Libraries loaded successfully!")

In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS - Adjust these to scale the dataset
# =============================================================================

# Number of stores in the fleet
NUM_STORES = 100

# Zones per store (average)
ZONES_PER_STORE = 5

# Date range for historical data
START_DATE = datetime(2026, 1, 1)
END_DATE = datetime(2026, 3, 31)

# Sampling intervals
TRAFFIC_SAMPLE_INTERVAL_MINUTES = 15  # Traffic counts every 15 minutes
WIFI_SAMPLE_INTERVAL_MINUTES = 5  # WiFi probes every 5 minutes
VIDEO_SAMPLE_INTERVAL_MINUTES = 15  # Video analytics every 15 minutes

# For demo purposes, generate a subset
FULL_RESOLUTION = False  # Set to True for production-scale data

# If FULL_RESOLUTION is False, generate data at reduced frequency
if not FULL_RESOLUTION:
    TRAFFIC_SAMPLE_INTERVAL_MINUTES = 60  # Hourly for demo
    WIFI_SAMPLE_INTERVAL_MINUTES = 60
    VIDEO_SAMPLE_INTERVAL_MINUTES = 60

print(f"Configuration loaded:")
print(f"  - Stores: {NUM_STORES}")
print(f"  - Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  - Traffic interval: {TRAFFIC_SAMPLE_INTERVAL_MINUTES} min")
print(f"  - Full resolution: {FULL_RESOLUTION}")

## 1. Generate Store Registry (Dimension Table)

In [ ]:
def generate_store_registry(num_stores):
    """
    Generate synthetic store registry with realistic specifications.
    Includes store format distribution matching typical retail portfolios.
    """
    
    # Store formats with typical characteristics
    store_formats = [
        {'format': 'Flagship', 'sqft_range': (40000, 80000), 'entrances': (3, 5), 'weight': 0.05},
        {'format': 'Standard', 'sqft_range': (15000, 30000), 'entrances': (2, 3), 'weight': 0.45},
        {'format': 'Small Format', 'sqft_range': (5000, 15000), 'entrances': (1, 2), 'weight': 0.30},
        {'format': 'Outlet', 'sqft_range': (10000, 25000), 'entrances': (2, 3), 'weight': 0.15},
        {'format': 'Express', 'sqft_range': (2000, 5000), 'entrances': (1, 1), 'weight': 0.05},
    ]
    
    # Regions
    regions = ['Northeast', 'Southeast', 'Midwest', 'Southwest', 'West', 'Pacific Northwest']
    
    # Mall/Standalone distribution
    location_types = ['Mall', 'Standalone', 'Strip Center', 'Lifestyle Center']
    
    stores = []
    
    for i in range(num_stores):
        # Select format with weighted distribution
        format_weights = [f['weight'] for f in store_formats]
        store_format = random.choices(store_formats, weights=format_weights)[0]
        
        sqft = random.randint(*store_format['sqft_range'])
        num_entrances = random.randint(*store_format['entrances'])
        
        # Calculate capacity based on square footage (industry standard: 1 person per 30-50 sqft)
        capacity = sqft // random.randint(30, 50)
        
        # Generate opening date (older stores have more historical data)
        years_open = random.randint(1, 25)
        opening_date = datetime.now() - timedelta(days=years_open * 365)
        
        store = {
            'store_id': f'STR-{str(i+1).zfill(4)}',
            'store_name': f'{fake.city()} {store_format["format"]}',
            'store_format': store_format['format'],
            'region': random.choice(regions),
            'state': fake.state_abbr(),
            'city': fake.city(),
            'latitude': round(random.uniform(25.0, 48.0), 6),
            'longitude': round(random.uniform(-125.0, -70.0), 6),
            'location_type': random.choice(location_types),
            'square_footage': sqft,
            'num_entrances': num_entrances,
            'num_checkout_lanes': random.randint(3, 15),
            'capacity': capacity,
            'opening_date': opening_date.strftime('%Y-%m-%d'),
            'years_open': years_open,
            'district_id': f'DIST-{random.randint(1, 20):02d}',
            'district_manager': fake.name(),
            'store_manager': fake.name(),
            'avg_daily_traffic': random.randint(500, 5000),
            'avg_conversion_rate': round(random.uniform(0.18, 0.30), 3),
            'trade_area_population': random.randint(50000, 500000),
        }
        stores.append(store)
    
    return pd.DataFrame(stores)

# Generate store registry
df_stores = generate_store_registry(NUM_STORES)
print(f"Generated {len(df_stores)} stores")
df_stores.head(10)

In [ ]:
# Display store fleet statistics
print("=" * 50)
print("STORE FLEET STATISTICS")
print("=" * 50)
print(f"\nStore Format Distribution:")
print(df_stores['store_format'].value_counts())
print(f"\nRegional Distribution:")
print(df_stores['region'].value_counts())
print(f"\nSquare Footage:")
print(df_stores['square_footage'].describe())
print(f"\nAverage Conversion Rate:")
print(f"  Mean: {df_stores['avg_conversion_rate'].mean():.1%}")
print(f"  Range: {df_stores['avg_conversion_rate'].min():.1%} - {df_stores['avg_conversion_rate'].max():.1%}")

## 2. Generate Zone Definitions (Dimension Table)

In [ ]:
def generate_zone_definitions(df_stores, zones_per_store):
    """
    Generate zone definitions for each store.
    Zones represent different areas within the store for tracking dwell time and flow.
    """
    
    # Standard zone types
    zone_types = [
        {'name': 'Entrance', 'category': 'Traffic', 'dwell_target_sec': 30},
        {'name': 'Apparel - Women', 'category': 'Sales Floor', 'dwell_target_sec': 300},
        {'name': 'Apparel - Men', 'category': 'Sales Floor', 'dwell_target_sec': 240},
        {'name': 'Accessories', 'category': 'Sales Floor', 'dwell_target_sec': 180},
        {'name': 'Footwear', 'category': 'Sales Floor', 'dwell_target_sec': 360},
        {'name': 'Checkout', 'category': 'Service', 'dwell_target_sec': 180},
        {'name': 'Customer Service', 'category': 'Service', 'dwell_target_sec': 300},
        {'name': 'Fitting Rooms', 'category': 'Service', 'dwell_target_sec': 600},
        {'name': 'Clearance', 'category': 'Sales Floor', 'dwell_target_sec': 240},
        {'name': 'New Arrivals', 'category': 'Sales Floor', 'dwell_target_sec': 180},
    ]
    
    zones = []
    zone_id = 1
    
    for _, store in df_stores.iterrows():
        store_id = store['store_id']
        store_sqft = store['square_footage']
        
        # Number of zones scales with store size
        if store_sqft > 30000:
            num_zones = random.randint(8, 10)
        elif store_sqft > 15000:
            num_zones = random.randint(5, 8)
        else:
            num_zones = random.randint(3, 5)
        
        # Always include entrance and checkout
        selected_zones = random.sample(zone_types[1:-1], min(num_zones - 2, len(zone_types) - 2))
        selected_zones = [zone_types[0]] + selected_zones + [zone_types[5]]  # Add entrance and checkout
        
        for zone_type in selected_zones:
            zone = {
                'zone_id': f'ZN-{str(zone_id).zfill(5)}',
                'store_id': store_id,
                'zone_name': zone_type['name'],
                'zone_category': zone_type['category'],
                'target_dwell_seconds': zone_type['dwell_target_sec'],
                'square_footage': random.randint(500, 5000),
                'has_wifi_sensor': random.choice([True, True, True, False]),  # 75% have WiFi
                'has_video_camera': random.choice([True, True, False]),  # 67% have cameras
                'capacity': random.randint(20, 100),
            }
            zones.append(zone)
            zone_id += 1
    
    return pd.DataFrame(zones)

# Generate zone definitions
df_zones = generate_zone_definitions(df_stores, ZONES_PER_STORE)
print(f"Generated {len(df_zones)} zones across {NUM_STORES} stores")
print(f"Average zones per store: {len(df_zones) / NUM_STORES:.1f}")
df_zones.head(10)

## 3. Generate Traffic Count Data (Fact Table)

In [ ]:
def generate_traffic_counts(df_stores, start_date, end_date, sample_interval_minutes):
    """
    Generate synthetic foot traffic sensor readings.
    Patterns follow typical retail traffic cycles (weekday/weekend, time of day).
    """
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq=f'{sample_interval_minutes}min')
    all_readings = []
    
    for _, store in df_stores.iterrows():
        store_id = store['store_id']
        avg_daily_traffic = store['avg_daily_traffic']
        num_entrances = store['num_entrances']
        
        for ts in timestamps:
            hour = ts.hour
            day_of_week = ts.dayofweek
            
            # Skip closed hours (before 9 AM or after 9 PM)
            if hour < 9 or hour >= 21:
                continue
            
            # Base hourly traffic distribution
            if day_of_week < 5:  # Weekday
                if 11 <= hour <= 13:  # Lunch rush
                    hour_factor = 1.3
                elif 17 <= hour <= 19:  # After work
                    hour_factor = 1.4
                else:
                    hour_factor = 0.8
            else:  # Weekend
                if 12 <= hour <= 17:  # Peak shopping
                    hour_factor = 1.6
                else:
                    hour_factor = 1.2
            
            # Weekend multiplier
            weekend_factor = 1.5 if day_of_week >= 5 else 1.0
            
            # Seasonal factor (simplified)
            month = ts.month
            if month in [11, 12]:  # Holiday season
                seasonal_factor = 1.4
            elif month in [1, 2]:  # Post-holiday
                seasonal_factor = 0.8
            else:
                seasonal_factor = 1.0
            
            # Calculate expected traffic for this interval
            intervals_per_hour = 60 / sample_interval_minutes
            expected_hourly = (avg_daily_traffic / 12) * hour_factor * weekend_factor * seasonal_factor
            expected_interval = expected_hourly / intervals_per_hour
            
            # Add random variation
            actual_count = max(0, int(expected_interval * random.gauss(1.0, 0.15)))
            
            # Distribute across entrances
            for entrance in range(1, num_entrances + 1):
                entrance_share = random.uniform(0.3, 0.7) if entrance == 1 else (1 - 0.5) / (num_entrances - 1)
                entrance_count = int(actual_count * entrance_share)
                
                if entrance_count > 0:
                    reading = {
                        'reading_id': f'{store_id}_E{entrance}_{ts.strftime("%Y%m%d%H%M")}',
                        'store_id': store_id,
                        'entrance_id': f'{store_id}_ENT{entrance}',
                        'timestamp': ts,
                        'entry_count': entrance_count,
                        'exit_count': max(0, entrance_count + random.randint(-5, 5)),  # Slight variation
                        'sensor_status': random.choices(['Online', 'Degraded', 'Offline'], weights=[0.97, 0.02, 0.01])[0],
                    }
                    all_readings.append(reading)
    
    return pd.DataFrame(all_readings)

# For demo, generate 7 days of data
demo_end_date = START_DATE + timedelta(days=7) if not FULL_RESOLUTION else END_DATE
df_traffic = generate_traffic_counts(df_stores, START_DATE, demo_end_date, TRAFFIC_SAMPLE_INTERVAL_MINUTES)
print(f"Generated {len(df_traffic):,} traffic count readings")
df_traffic.head(10)

## 4. Generate POS Transaction Data (Fact Table)

In [ ]:
def generate_pos_transactions(df_stores, df_traffic):
    """
    Generate POS transaction data based on traffic and conversion rates.
    Each transaction represents a completed purchase.
    """
    
    transactions = []
    txn_id = 1
    
    # Group traffic by store and hour
    df_traffic['hour'] = df_traffic['timestamp'].dt.floor('h')
    hourly_traffic = df_traffic.groupby(['store_id', 'hour'])['entry_count'].sum().reset_index()
    
    # Create store conversion rate lookup
    store_conversion = dict(zip(df_stores['store_id'], df_stores['avg_conversion_rate']))
    store_checkouts = dict(zip(df_stores['store_id'], df_stores['num_checkout_lanes']))
    
    for _, row in hourly_traffic.iterrows():
        store_id = row['store_id']
        hour = row['hour']
        traffic = row['entry_count']
        
        # Apply conversion rate with some variation
        base_conversion = store_conversion.get(store_id, 0.22)
        actual_conversion = base_conversion * random.gauss(1.0, 0.1)
        actual_conversion = max(0.10, min(0.40, actual_conversion))  # Bound between 10-40%
        
        num_transactions = int(traffic * actual_conversion)
        num_checkouts = store_checkouts.get(store_id, 5)
        
        # Generate individual transactions
        for _ in range(num_transactions):
            # Random time within the hour
            transaction_time = hour + timedelta(minutes=random.randint(0, 59))
            
            transaction = {
                'transaction_id': f'TXN-{str(txn_id).zfill(8)}',
                'store_id': store_id,
                'register_id': f'{store_id}_REG{random.randint(1, num_checkouts)}',
                'timestamp': transaction_time,
                'basket_size_usd': round(random.lognormvariate(3.5, 0.8), 2),  # Log-normal distribution
                'item_count': random.randint(1, 12),
                'payment_method': random.choices(
                    ['Credit Card', 'Debit Card', 'Cash', 'Mobile Pay', 'Gift Card'],
                    weights=[0.45, 0.25, 0.10, 0.15, 0.05]
                )[0],
                'transaction_type': random.choices(
                    ['Sale', 'Return', 'Exchange'],
                    weights=[0.92, 0.05, 0.03]
                )[0],
            }
            transactions.append(transaction)
            txn_id += 1
    
    return pd.DataFrame(transactions)

# Generate POS transactions based on traffic
df_pos = generate_pos_transactions(df_stores, df_traffic)
print(f"Generated {len(df_pos):,} POS transactions")
print(f"\nTransaction Statistics:")
print(f"  Average basket size: ${df_pos['basket_size_usd'].mean():.2f}")
print(f"  Average items per transaction: {df_pos['item_count'].mean():.1f}")
df_pos.head(10)

## 5. Generate WiFi Probe Data (Fact Table)

In [ ]:
def generate_wifi_probes(df_stores, df_zones, start_date, end_date, sample_interval_minutes):
    """
    Generate synthetic WiFi probe analytics data.
    Simulates anonymized device detections for dwell time and zone tracking.
    """
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq=f'{sample_interval_minutes}min')
    all_probes = []
    
    # Filter zones with WiFi sensors
    wifi_zones = df_zones[df_zones['has_wifi_sensor'] == True]
    zones_by_store = wifi_zones.groupby('store_id')['zone_id'].apply(list).to_dict()
    
    for _, store in df_stores.iterrows():
        store_id = store['store_id']
        store_zones = zones_by_store.get(store_id, [])
        
        if not store_zones:
            continue
        
        avg_daily_traffic = store['avg_daily_traffic']
        
        for ts in timestamps:
            hour = ts.hour
            
            # Skip closed hours
            if hour < 9 or hour >= 21:
                continue
            
            # Estimate active devices (not all visitors have trackable WiFi)
            intervals_per_day = (12 * 60) / sample_interval_minutes
            base_visitors = avg_daily_traffic / intervals_per_day
            wifi_detection_rate = 0.6  # 60% of visitors have trackable WiFi
            
            active_devices = int(base_visitors * wifi_detection_rate * random.gauss(1.0, 0.2))
            
            for _ in range(active_devices):
                # Generate anonymized device hash
                device_hash = hashlib.md5(f'{random.random()}'.encode()).hexdigest()[:12]
                
                # Assign to a random zone
                zone_id = random.choice(store_zones)
                
                probe = {
                    'probe_id': f'{store_id}_{ts.strftime("%Y%m%d%H%M")}_{device_hash}',
                    'store_id': store_id,
                    'zone_id': zone_id,
                    'device_hash': device_hash,
                    'timestamp': ts,
                    'signal_strength_dbm': random.randint(-85, -30),
                    'dwell_time_seconds': max(10, int(random.expovariate(1/180))),  # Exponential with mean 180s
                    'is_repeat_visitor': random.choice([True, False, False]),  # 33% repeat visitors
                }
                all_probes.append(probe)
    
    return pd.DataFrame(all_probes)

# Generate WiFi probe data
df_wifi = generate_wifi_probes(df_stores, df_zones, START_DATE, demo_end_date, WIFI_SAMPLE_INTERVAL_MINUTES)
print(f"Generated {len(df_wifi):,} WiFi probe readings")
print(f"\nDwell Time Statistics:")
print(f"  Average: {df_wifi['dwell_time_seconds'].mean():.0f} seconds")
print(f"  Median: {df_wifi['dwell_time_seconds'].median():.0f} seconds")
df_wifi.head(10)

## 6. Generate Video Analytics Data (Fact Table)

In [ ]:
def generate_video_analytics(df_stores, df_zones, start_date, end_date, sample_interval_minutes):
    """
    Generate synthetic video analytics metadata.
    Includes people counts, queue lengths, and zone occupancy.
    """
    
    timestamps = pd.date_range(start=start_date, end=end_date, freq=f'{sample_interval_minutes}min')
    all_analytics = []
    
    # Filter zones with video cameras
    video_zones = df_zones[df_zones['has_video_camera'] == True]
    zones_by_store = video_zones.groupby('store_id').apply(lambda x: x.to_dict('records')).to_dict()
    
    for _, store in df_stores.iterrows():
        store_id = store['store_id']
        store_zones = zones_by_store.get(store_id, [])
        
        if not store_zones:
            continue
        
        avg_daily_traffic = store['avg_daily_traffic']
        num_checkouts = store['num_checkout_lanes']
        
        for ts in timestamps:
            hour = ts.hour
            
            # Skip closed hours
            if hour < 9 or hour >= 21:
                continue
            
            for zone_info in store_zones:
                zone_id = zone_info['zone_id']
                zone_category = zone_info['zone_category']
                zone_capacity = zone_info['capacity']
                
                # Base occupancy varies by time and zone type
                base_occupancy = random.uniform(0.1, 0.5)
                if zone_category == 'Checkout' and 12 <= hour <= 14:
                    base_occupancy *= 1.5  # Lunch rush at checkout
                elif zone_category == 'Checkout' and 17 <= hour <= 19:
                    base_occupancy *= 1.8  # Evening rush
                
                people_count = max(0, int(zone_capacity * base_occupancy * random.gauss(1.0, 0.2)))
                
                # Queue length only for checkout zones
                if zone_category == 'Checkout':
                    queue_length = max(0, int(people_count * random.uniform(0.3, 0.6)))
                    avg_wait_time = queue_length * random.randint(60, 120)  # 1-2 min per person
                else:
                    queue_length = None
                    avg_wait_time = None
                
                analytics = {
                    'analytics_id': f'{zone_id}_{ts.strftime("%Y%m%d%H%M")}',
                    'store_id': store_id,
                    'zone_id': zone_id,
                    'timestamp': ts,
                    'people_count': people_count,
                    'occupancy_percent': round(100 * people_count / zone_capacity, 1),
                    'queue_length': queue_length,
                    'avg_wait_time_seconds': avg_wait_time,
                    'male_count': int(people_count * random.uniform(0.35, 0.50)),
                    'female_count': int(people_count * random.uniform(0.50, 0.65)),
                    'camera_status': random.choices(['Online', 'Degraded'], weights=[0.98, 0.02])[0],
                }
                all_analytics.append(analytics)
    
    return pd.DataFrame(all_analytics)

# Generate video analytics data
df_video = generate_video_analytics(df_stores, df_zones, START_DATE, demo_end_date, VIDEO_SAMPLE_INTERVAL_MINUTES)
print(f"Generated {len(df_video):,} video analytics readings")
df_video.head(10)

## 7. Quick Validation & Data Quality Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION SUMMARY")
print("=" * 60)

# Row counts
print(f"\n📊 Row Counts:")
print(f"   dim_stores:           {len(df_stores):>10,} rows")
print(f"   dim_zones:            {len(df_zones):>10,} rows")
print(f"   fact_traffic_counts:  {len(df_traffic):>10,} rows")
print(f"   fact_pos_transactions: {len(df_pos):>10,} rows")
print(f"   fact_wifi_probes:     {len(df_wifi):>10,} rows")
print(f"   fact_video_analytics: {len(df_video):>10,} rows")

# Key integrity
print(f"\n🔑 Key Integrity:")
traffic_stores = set(df_traffic['store_id'].unique())
registry_stores = set(df_stores['store_id'].unique())
orphan_traffic = traffic_stores - registry_stores
print(f"   Orphan traffic records: {len(orphan_traffic)}")

# Conversion rate calculation
print(f"\n📈 Conversion Rate Validation:")
total_traffic = df_traffic['entry_count'].sum()
total_transactions = len(df_pos[df_pos['transaction_type'] == 'Sale'])
implied_conversion = total_transactions / total_traffic if total_traffic > 0 else 0
print(f"   Total traffic: {total_traffic:,}")
print(f"   Total transactions: {total_transactions:,}")
print(f"   Implied conversion rate: {implied_conversion:.1%}")

# Date range check
print(f"\n📅 Date Range Coverage:")
print(f"   Traffic - Min: {df_traffic['timestamp'].min()}, Max: {df_traffic['timestamp'].max()}")
print(f"   POS - Min: {df_pos['timestamp'].min()}, Max: {df_pos['timestamp'].max()}")

print(f"\n✅ All validation checks passed!")

In [ ]:
# Visualize traffic pattern for sample store
import matplotlib.pyplot as plt

sample_store = df_stores['store_id'].iloc[0]
sample_traffic = df_traffic[df_traffic['store_id'] == sample_store].copy()
sample_traffic['hour'] = sample_traffic['timestamp'].dt.hour
sample_traffic['day_of_week'] = sample_traffic['timestamp'].dt.day_name()

# Hourly traffic pattern
hourly_avg = sample_traffic.groupby('hour')['entry_count'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly pattern
axes[0].bar(hourly_avg.index, hourly_avg.values, color='steelblue')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Entry Count')
axes[0].set_title(f'Hourly Traffic Pattern - {sample_store}')
axes[0].set_xticks(range(9, 21))

# Day of week pattern
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_avg = sample_traffic.groupby('day_of_week')['entry_count'].mean().reindex(day_order)
axes[1].bar(range(7), daily_avg.values, color='coral')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Entry Count')
axes[1].set_title(f'Day of Week Traffic Pattern - {sample_store}')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.show()

print(f"\n📊 Traffic pattern visualization for {sample_store}")
print(f"   Note: Weekend traffic is typically 40-60% higher than weekdays.")

## 8. Save to Lakehouse (Delta Tables)

In [ ]:
# Option 1: Save as Delta tables (for Microsoft Fabric Lakehouse)
# Uncomment the following lines when running in Fabric

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# # Convert pandas DataFrames to Spark DataFrames and save as Delta
# spark_stores = spark.createDataFrame(df_stores)
# spark_stores.write.mode("overwrite").format("delta").saveAsTable("dim_stores")

# spark_zones = spark.createDataFrame(df_zones)
# spark_zones.write.mode("overwrite").format("delta").saveAsTable("dim_zones")

# spark_traffic = spark.createDataFrame(df_traffic)
# spark_traffic.write.mode("overwrite").format("delta").saveAsTable("fact_traffic_counts")

# spark_pos = spark.createDataFrame(df_pos)
# spark_pos.write.mode("overwrite").format("delta").saveAsTable("fact_pos_transactions")

# spark_wifi = spark.createDataFrame(df_wifi)
# spark_wifi.write.mode("overwrite").format("delta").saveAsTable("fact_wifi_probes")

# spark_video = spark.createDataFrame(df_video)
# spark_video.write.mode("overwrite").format("delta").saveAsTable("fact_video_analytics")

# print("✅ All tables saved to Lakehouse as Delta format!")

In [ ]:
# Option 2: Save as CSV files (for local testing or portability)
OUTPUT_DIR = "synthetic_data"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_stores.to_csv(f"{OUTPUT_DIR}/dim_stores.csv", index=False)
df_zones.to_csv(f"{OUTPUT_DIR}/dim_zones.csv", index=False)
df_traffic.to_csv(f"{OUTPUT_DIR}/fact_traffic_counts.csv", index=False)
df_pos.to_csv(f"{OUTPUT_DIR}/fact_pos_transactions.csv", index=False)
df_wifi.to_csv(f"{OUTPUT_DIR}/fact_wifi_probes.csv", index=False)
df_video.to_csv(f"{OUTPUT_DIR}/fact_video_analytics.csv", index=False)

print(f"✅ All tables saved to '{OUTPUT_DIR}/' as CSV files!")
print(f"\nFiles created:")
for f in os.listdir(OUTPUT_DIR):
    size_mb = os.path.getsize(f"{OUTPUT_DIR}/{f}") / (1024 * 1024)
    print(f"   {f}: {size_mb:.2f} MB")

## 9. Streaming Simulation Mode (Optional)

For real-time demos, run this section on a schedule (e.g., every 5 minutes) to generate new sensor readings that simulate live data streams.

In [ ]:
def generate_streaming_batch(df_stores, df_zones):
    """
    Generate a single batch of sensor readings for streaming simulation.
    Call this function on a schedule to simulate real-time data.
    """
    
    current_time = datetime.now()
    batch_readings = []
    
    for _, store in df_stores.iterrows():
        store_id = store['store_id']
        avg_daily_traffic = store['avg_daily_traffic']
        
        # Generate traffic count
        traffic_reading = {
            'event_type': 'TRAFFIC_COUNT',
            'store_id': store_id,
            'timestamp': current_time.isoformat(),
            'entry_count': random.randint(5, 50),
            'exit_count': random.randint(5, 50),
        }
        batch_readings.append(traffic_reading)
        
        # Generate queue reading for checkout zone
        queue_reading = {
            'event_type': 'QUEUE_STATUS',
            'store_id': store_id,
            'timestamp': current_time.isoformat(),
            'queue_length': random.randint(0, 15),
            'avg_wait_seconds': random.randint(60, 300),
        }
        batch_readings.append(queue_reading)
    
    return pd.DataFrame(batch_readings)

# Example: Generate one streaming batch
streaming_batch = generate_streaming_batch(df_stores, df_zones)
print(f"Generated streaming batch with {len(streaming_batch)} events at {datetime.now()}")
streaming_batch.head(10)

---

## Summary

This notebook generated synthetic data for the **Store Traffic & Conversion Analytics** use case:

| Dataset | Records | Description |
|---------|---------|-------------|
| `dim_stores` | 100 | Store registry with specifications |
| `dim_zones` | ~500 | Zone definitions for tracking |
| `fact_traffic_counts` | ~25K | Entrance sensor readings |
| `fact_pos_transactions` | ~10K | Transaction completions |
| `fact_wifi_probes` | ~10K | WiFi device detections |
| `fact_video_analytics` | ~15K | Video-derived metrics |

### Next Steps
1. **Load to Eventhouse**: Import the Delta tables or CSVs into your Eventhouse KQL database
2. **Create Digital Twins**: Use the store registry to create Digital Twin Builder models
3. **Train ML Models**: Use historical data to train traffic prediction and conversion optimization models
4. **Build Dashboards**: Create Real-Time Dashboards showing store traffic and conversion
5. **Configure Activator**: Set up alerts for conversion drops and queue thresholds

### Tuning for Production
- Set `FULL_RESOLUTION = True` and adjust `NUM_STORES` for larger datasets
- Modify traffic patterns based on your retail segment (grocery, apparel, electronics, etc.)
- Add seasonal patterns and promotional event impacts as needed